# ⬇️ Download Stream

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/nb/downloader-m3u8/guides/video/Download_Stream.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

def process_m3u8(stream_url, output_file):
    # Download the M3U8 file
    input_m3u8_path = 'input.m3u8'
    output_m3u8_path = 'output.m3u8'
    download_dir = 'downloaded_segments/'
    num_threads = 100

    response = requests.get(stream_url)
    with open(input_m3u8_path, 'wb') as file:
        file.write(response.content)

    # Ensure the download directory exists
    if not os.path.exists(download_dir):
        os.makedirs(download_dir)

    def generate_filename(url):
        """Generate a unique filename for each URL based on its hash."""
        url_hash = hashlib.md5(url.encode()).hexdigest()
        return f"{url_hash}.ts"

    def download_segment(url, file_path):
        """Download a segment from the URL to the given file path."""
        try:
            response = requests.get(url, stream=True)
            if response.status_code == 200:
                with open(file_path, 'wb') as f:
                    for chunk in response.iter_content(1024):
                        f.write(chunk)
                return file_path
            else:
                print(f"Failed to download {url}")
                return None
        except Exception as e:
            print(f"Error downloading {url}: {e}")
            return None

    # Read the input M3U8 file
    with open(input_m3u8_path, 'r') as file:
        lines = file.readlines()

    # Collect all segment URLs and their intended local file paths
    downloads = []
    for line in lines:
        if line.startswith("https://"):
            filename = generate_filename(line.strip())
            local_path = os.path.join(download_dir, filename)
            downloads.append((line.strip(), local_path))

    # Use ThreadPoolExecutor to download files concurrently
    def process_downloads():
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            future_to_url = {executor.submit(download_segment, url, path): url for url, path in downloads}
            for future in as_completed(future_to_url):
                url = future_to_url[future]
                try:
                    path = future.result()
                    if path:
                        print(f"Downloaded {path}")
                except Exception as e:
                    print(f"{url} generated an exception: {e}")

    # Perform downloads
    process_downloads()

    # Write the output M3U8 file with the updated local paths
    with open(output_m3u8_path, 'w') as file:
        for line in lines:
            if line.startswith("https://"):
                filename = generate_filename(line.strip())
                local_path = os.path.join(download_dir, filename)
                file.write(local_path + '\n')
            else:
                file.write(line)

    print("M3U8 file processing complete. All segments downloaded and file paths updated.")

    # Convert the updated M3U8 file to an MP4 file using FFmpeg
    subprocess.run(['ffmpeg', '-i', output_m3u8_path, '-c', 'copy', '-bsf:a', 'aac_adtstoasc', output_file])

    print(f"Conversion complete. Output file: {output_file}")



In [ ]:
stream_url = "YOUR_URL_HERE"
output_file = "output.mp4"
process_m3u8(stream_url, output_file)